# YAML - Rust

All 8 Rust examples from [docs/yaml.md](https://platob.github.io/yggdryl/yaml/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

In [ ]:
use yggdryl::yaml;

let value = yaml::from_str("symbol: AAPL\nquantity: 2\n")?;
assert_eq!(
    value.get_key_str("symbol").and_then(|value| value.as_str()),
    Some("AAPL"),
);

let encoded = yaml::to_vec(&value)?;
assert_eq!(yaml::from_slice(&encoded)?, value);

// One document means one.
assert!(yaml::from_str("id: 1\n---\nid: 2\n").is_err());

## Documents

In [ ]:
use yggdryl::{Value, yaml};

let documents = yaml::from_str_all("id: 1\n---\nid: 2\n---\nnull\n")?;
assert_eq!(documents.len(), 3);
assert_eq!(documents[2], Value::Null);

let encoded = yaml::to_vec_all(&documents)?;
assert!(std::str::from_utf8(&encoded)?.contains("\n---\n"));
assert_eq!(yaml::from_slice_all(&encoded)?, documents);

In [ ]:
use std::io::Cursor;
use yggdryl::{Value, yaml};

let mut reader = yaml::Reader::new(Cursor::new("id: 1\n---\nitems: [1, 2\n"));

let first = reader.next().expect("a first document")?;
assert_eq!(first.get_key_str("id"), Some(&Value::U64(1)));
assert!(reader.byte_offset() >= "id: 1\n".len());

// The second document is malformed, and the reader is done after saying so.
assert!(reader.next().expect("a second document").is_err());
assert!(reader.next().is_none());

## Laying out a dump

In [ ]:
use yggdryl::generic::Value;
use yggdryl::text::Formatting;

let value = Value::from_mapping([
    (Value::String("id".into()), Value::I64(1)),
    (Value::String("tags".into()), Value::from_sequence([Value::String("a".into())])),
])?;

assert_eq!(yggdryl::yaml::to_vec(&value)?, b"id: 1\ntags:\n  - a\n");
assert_eq!(
    yggdryl::yaml::to_vec_with_formatting(&value, Formatting::indented(4))?,
    b"id: 1\ntags:\n    - a\n",
);

// Flow style is the explicit opt-in, and it round-trips.
let flow = yggdryl::yaml::to_vec_with_formatting(&value, Formatting::compact())?;
assert_eq!(flow, b"{id: 1, tags: [a]}\n");
assert_eq!(yggdryl::yaml::from_slice(&flow)?, value);

## Tags are read, never written

In [ ]:
use yggdryl::{Value, yaml};

let value = Value::from_mapping([
    (Value::from("payload"), Value::from(vec![0_u8, 255])),
])?;

let encoded = yaml::to_vec(&value)?;
let text = std::str::from_utf8(&encoded)?;
assert!(!text.contains("!yggdryl"));
assert!(text.contains(r#""$yggdryl": "bytes""#));

assert_eq!(yaml::from_slice(&encoded)?, value);

In [ ]:
use yggdryl::{Value, yaml};

// A machine tag on input is semantic: it names a kind the value model has.
assert_eq!(
    yaml::from_str("!yggdryl/bytes AP8=\n")?,
    Value::from(vec![0_u8, 255]),
);

// An application tag names nothing the value model has, so it stays the
// annotation YAML defines it to be and the node under it is the value.
let value = yaml::from_str("!vendor:quantity {value: 4}\n")?;
assert_eq!(value.get_key_str("value"), Some(&Value::U64(4)));

// A comment is not read either.
assert_eq!(
    yaml::from_str("# vendor:attacker\n!vendor:quantity {value: 4}\n")?,
    value,
);

In [ ]:
use yggdryl::{Value, yaml};

let collision = Value::from_mapping([
    (Value::from("$yggdryl"), Value::from("bytes")),
    (Value::from("value"), Value::from("AP8=")),
])?;

let encoded = yaml::to_vec(&collision)?;
assert!(std::str::from_utf8(&encoded)?.contains(r#""$yggdryl": "mapping""#));
assert_eq!(yaml::from_slice(&encoded)?, collision);

## Placeholders, and the quoting that YAML requires

In [ ]:
use yggdryl::text::{Format, Loading, Placeholders};
use yggdryl::Value;

let loading = Loading::new()
    .with_placeholders(Placeholders::new().with_variable("PORT", Value::I64(8080)));

// Quoted: a string scalar, so it resolves - and adopts the resolved type.
let quoted = yggdryl::text::from_str_with("port: \"{{ PORT }}\"\n", Format::Yaml, &loading)?;
assert_eq!(quoted.get_key_str("port"), Some(&Value::I64(8080)));

// Unquoted: the flow mapping YAML read, untouched.
let bare = yggdryl::text::from_str_with("port: {{ PORT }}\n", Format::Yaml, &loading)?;
assert!(bare.get_key_str("port").and_then(Value::as_mapping).is_some());